# 04. Data Manipulation, Filtering & Built-in Functions

In real-world data engineering, data is messy and needs filtering, cleaning, conditional transformations, and formatting.

### Topics Covered:
1. **Filtering Rows (`filter` and `where`) with single & compound conditions**
2. **Conditional Logic with `when()` and `otherwise()` (PySpark's IF-THEN-ELSE)**
3. **Handling Missing / Null Values (`isNull`, `isNotNull`, `dropna`, `fillna`)**
4. **Handling Duplicates (`distinct` vs `dropDuplicates`)**
5. **String Functions (`upper`, `lower`, `trim`, `concat_ws`, `regexp_replace`)**
6. **Date & Timestamp Functions (`current_date`, `datediff`, `date_add`, `year`)**
7. **Complex Types: Arrays (`split`, `explode`, `array_contains`)**
8. **Sorting Data (`orderBy` and `sort`)**
9. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, upper, lower, trim, concat, concat_ws,
    current_date, datediff, date_add, to_date, year, month,
    split, explode, array_contains, desc, asc
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

spark = SparkSession.builder \
    .appName("04_Data_Manipulation") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession is running!")


---
## 2. Creating Sample Dataset with Realistic Data

We'll create a customer orders dataset with some missing values, mixed casings, and dates.


In [ ]:
raw_data = [
    (101, "  alice smith  ", "Electronics", 450.0, "2023-01-15", "laptop,mouse"),
    (102, "Bob Johnson", "Clothing", 85.5, "2023-02-10", "shirt,jeans"),
    (103, "charlie lee", "Electronics", None, "2023-03-05", "headphones"),
    (104, "Diana Prince", None, 320.0, "2023-04-20", "phone,case,charger"),
    (105, "Evan Wright", "Groceries", 45.0, "2023-05-12", "apples,bread,milk"),
    (106, "alice smith", "Electronics", 450.0, "2023-01-15", "laptop,mouse"), # Duplicate row
    (107, "Fiona Gallagher", "Clothing", 150.0, None, "jacket"),
    (108, "George Clark", "Books", 25.0, "2023-06-18", "novel")
]

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("items", StringType(), True)
])

df = spark.createDataFrame(raw_data, schema=schema)
df.show(truncate=False)


---
## 3. Filtering Rows (`filter` and `where`)

* Note: `filter()` and `where()` are identical in PySpark; `where()` is provided for SQL familiarity.
* **Important syntax rule**: In PySpark, when combining conditions with AND (`&`) or OR (`|`), **each condition must be enclosed in parentheses `( )`**!


In [ ]:
# A. Single condition: orders greater than $100
df.filter(col("amount") > 100).show()

# B. Compound condition with & (AND): Electronics orders over $300
df.filter((col("category") == "Electronics") & (col("amount") > 300)).show()

# C. Compound condition with | (OR): Electronics OR Books
df.where((col("category") == "Electronics") | (col("category") == "Books")).show()

# D. Membership check using isin(): Category in list
df.filter(col("category").isin("Clothing", "Groceries")).show()


---
## 4. Conditional Logic with `when()` and `otherwise()`

This is PySpark's equivalent of `CASE WHEN ... THEN ... ELSE` in SQL or `IF/ELSE` in Python.
You can chain multiple `when()` calls for multiple cases.


In [ ]:
# Add a customer tier column based on amount
df_tier = df.withColumn(
    "spending_tier",
    when(col("amount") >= 300, "High Spender")
    .when((col("amount") >= 50) & (col("amount") < 300), "Medium Spender")
    .when(col("amount") < 50, "Low Spender")
    .otherwise("Unknown") # For null or unspecified amounts
)

df_tier.select("customer_name", "category", "amount", "spending_tier").show(truncate=False)


---
## 5. Handling Missing / Null Data

Nulls can cause errors or skewed analytics if not handled properly.

* `isNull()` / `isNotNull()`: Check if values are null.
* `dropna()`: Remove rows containing nulls.
* `fillna()`: Replace nulls with default fallback values.


In [ ]:
# A. Find rows where amount is null
print("Rows where amount is null:")
df.filter(col("amount").isNull()).show()

# B. Drop rows with null values:
# how="any" drops if ANY column is null; how="all" drops if ALL columns are null
# subset specifies which columns to consider
print("Dropping rows where amount OR category is null:")
df.dropna(how="any", subset=["amount", "category"]).show()

# C. Fill null values:
# Replace null amounts with 0.0, and null category with 'Uncategorized'
df_filled = df.fillna({"amount": 0.0, "category": "Uncategorized", "order_date": "1970-01-01"})
print("After filling nulls:")
df_filled.show()


---
## 6. Handling Duplicate Rows

* `distinct()`: Removes rows where **all** column values are identical.
* `dropDuplicates(subset)`: Removes rows that have duplicate values in specific key columns.


In [ ]:
print(f"Total row count before deduplication: {df.count()}")

# Remove duplicate rows based on customer_name and order_date
deduped_df = df.dropDuplicates(subset=["customer_name", "order_date"])
print(f"Total row count after deduplication: {deduped_df.count()}")
deduped_df.show()


---
## 7. String Functions (Cleaning Text)

PySpark provides a large library of string manipulation functions.


In [ ]:
cleaned_df = df_filled.withColumn(
    # Trim whitespace and convert to Title Case / Upper
    "clean_name", trim(upper(col("customer_name")))
).withColumn(
    # Format a friendly label
    "order_label", concat_ws(" - ", col("clean_name"), col("category"))
)

cleaned_df.select("clean_name", "category", "order_label").show(truncate=False)


---
## 8. Date and Time Functions

Working with dates is essential for analytics, time-series, and aggregations.


In [ ]:
date_df = df_filled.withColumn(
    # Convert string date column into standard DateType
    "order_date_dt", to_date(col("order_date"), "yyyy-MM-dd")
).withColumn(
    # Current date
    "today", current_date()
).withColumn(
    # Calculate days since order was placed
    "days_since_order", datediff(current_date(), col("order_date_dt"))
).withColumn(
    # Add 30-day return deadline
    "return_deadline", date_add(col("order_date_dt"), 30)
).withColumn(
    # Extract year and month
    "order_year", year(col("order_date_dt"))
)

date_df.select("order_id", "order_date_dt", "days_since_order", "return_deadline", "order_year").show()


---
## 9. Working with Arrays: `split`, `explode`, and `array_contains`

Many datasets have comma-separated values or nested arrays.


In [ ]:
# 1. split(): Convert comma-separated string into an array
array_df = df_filled.withColumn("items_list", split(col("items"), ","))
array_df.select("customer_name", "items_list").show(truncate=False)

# 2. array_contains(): Check if an array contains a specific item
laptop_buyers = array_df.filter(array_contains(col("items_list"), "laptop"))
print("Customers who purchased a laptop:")
laptop_buyers.select("customer_name", "items_list").show(truncate=False)

# 3. explode(): Flatten an array so each element gets its own row!
exploded_df = array_df.select("order_id", "customer_name", explode(col("items_list")).alias("single_item"))
print("Exploded individual items:")
exploded_df.show(10)


---
## 10. Sorting and Ordering Data (`orderBy` and `sort`)

* `sort()` and `orderBy()` are identical.
* Use `desc()` or `asc()` to specify sort direction.


In [ ]:
# Sort orders by amount descending (highest first)
df_filled.orderBy(desc("amount")).show()

# Sort by multiple columns: category ascending, then amount descending
df_filled.sort(asc("category"), desc("amount")).show()


---
## 11. Summary of Notebook 04
* Filter rows using `filter()` or `where()`; wrap each condition in `()`.
* Implement conditional logic with `when(condition, val).otherwise(default)`.
* Clean nulls with `dropna()` and `fillna()`.
* Deduplicate rows with `dropDuplicates(subset=[...])`.
* Manipulate strings, dates, and arrays using PySpark built-in functions.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [ ]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")
